Le but de cette partie est de rééquilibrer les données du dataset de fraudes à l'aide de call LLM.

In [1]:
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [huggingface_hub] [huggingface_hub]

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [ ]:
# il faut ensuite boucler sur toutes les lignes du dataset et appliquer les modèles de classification
from huggingface_hub import InferenceClient
import json
import os

os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxx"

client = InferenceClient(token=os.getenv("HF_TOKEN"))

messages = [
{
  "role": "user",
  "content": """
Tu es un expert en détection de fraude à l’assurance.

Ta tâche est de COMPLÉTER les champs manquants d’un enregistrement existant.
- N’invente rien d’extravagant
- Les valeurs doivent être réalistes et cohérentes avec le contexte
- Raisonne étape par étape mais ne montre PAS ton raisonnement
- Réponds uniquement avec un JSON valide, sans texte autour

### Exemple

Entrée :
{
    "Transaction_ID": "T3",
    "User_ID": "1860",
    "Transaction_Amount": 2395.02,
    "Transaction_Type": "ATM Withdrawal",
    "Time_of_Transaction": null,
    "Device_Used": "Mobile",
    "Location": "null",
    "Previous_Fraudulent_Transactions":3,
    "Account_Age": 115,
    "Number_of_Transactions_Last_24H": 9,
    "Payment_Method": "null",
    "Fraudulent": "0"
}

Sortie :
{
    "Transaction_ID": "T3",
    "User_ID": "1860",
    "Transaction_Amount": 2395.02,
    "Transaction_Type": "ATM Withdrawal",
    "Time_of_Transaction": 11.47734,
    "Device_Used": "Mobile",
    "Location": "Boston",
    "Previous_Fraudulent_Transactions":3,
    "Account_Age": 115,
    "Number_of_Transactions_Last_24H": 9,
    "Payment_Method": "Credit Card",
    "Fraudulent": "0"
}

Entrée :
{
    "Transaction_ID": "T112",
    "User_ID": "1455",
    "Transaction_Amount": null,
    "Transaction_Type": "Online Purchase",
    "Time_of_Transaction": 18.0,
    "Device_Used": "Mobile",
    "Location": "null",
    "Previous_Fraudulent_Transactions":1,
    "Account_Age": 116,
    "Number_of_Transactions_Last_24H": 11,
    "Payment_Method": "UPI",
    "Fraudulent": "0"
}

Sortie :
{
    "Transaction_ID": "T112",
    "User_ID": "1455",
    "Transaction_Amount": 2996.242497,
    "Transaction_Type": "Online Purchase",
    "Time_of_Transaction": 18.0,
    "Device_Used": "Mobile",
    "Location": "Seattle",
    "Previous_Fraudulent_Transactions": 1,
    "Account_Age": 116,
    "Number_of_Transactions_Last_24H": 11,
    "Payment_Method": "UPI",
    "Fraudulent": "0"
}

Voici l'enregistrement existant que tu dois compléter : 
Entrée :
{
    "Transaction_ID": "T50",
    "User_ID": "4005",
    "Transaction_Amount": null,
    "Transaction_Type": "ATM Withdrawal",
    "Time_of_Transaction": 4.0,
    "Device_Used": "Mobile",
    "Location": "Boston",
    "Previous_Fraudulent_Transactions":3,
    "Account_Age": 10,
    "Number_of_Transactions_Last_24H": 7,
    "Payment_Method": "Debit Card",
    "Fraudulent": "0"
}

"""
}
]


response = client.chat_completion(
    messages=messages,
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    max_tokens=300,
    temperature=0.5
)

generated_text = response.choices[0].message.content

# Extraire le JSON
try:
    json_start = generated_text.find('{')
    json_end = generated_text.rfind('}') + 1
    if json_start != -1 and json_end > json_start:
        json_str = generated_text[json_start:json_end]
        fraud_case = json.loads(json_str)
        print("\n=== Texte généré (JSON parsé) ===")
        print(json.dumps(fraud_case, indent=2, ensure_ascii=False))
    else:
        print("\n⚠️  Pas de JSON trouvé dans la réponse")
except Exception as e:
    print(f"\n❌ Erreur: {e}")

In [ ]:
from huggingface_hub import InferenceClient
import json
import os

os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxx"

client = InferenceClient(token=os.getenv("HF_TOKEN"))

messages = [
{
  "role": "user",
  "content": """
Tu es un expert en fraude à l’assurance automobile.

Ta tâche est de GÉNÉRER un cas de fraude réaliste.
- Le cas doit contenir des incohérences détectables
- Le sinistre doit sembler crédible mais suspect
- Raisonne étape par étape mais ne montre PAS ton raisonnement
- Réponds uniquement avec un JSON valide, sans texte autour
Tu dois remplir STRICTEMENT le JSON suivant :

{
  "Transaction_ID": "",
  "User_ID": "",
  "Transaction_Amount": 0.0,
  "Transaction_Type": "",
  "Time_of_Transaction": 0.0,
  "Device_Used": "",
  "Location": "",
  "Previous_Fraudulent_Transactions": 0,
  "Account_Age": 0,
  "Number_of_Transactions_Last_24H": 0,
  "Payment_Method": "",
  "Fraudulent": "1"
}

### Exemples

Exemple 1 :
{
    "Transaction_ID": "T28",
    "User_ID": "3433",
    "Transaction_Amount": 4519.04,
    "Transaction_Type": "Bill Payment",
    "Time_of_Transaction": 13.0,
    "Device_Used": "Tablet",
    "Location": "Boston",
    "Previous_Fraudulent_Transactions":0,
    "Account_Age": 81,
    "Number_of_Transactions_Last_24H": 10,
    "Payment_Method": "Debit Card",
    "Fraudulent": "1"
}

Exemple 2 :
{
    "Transaction_ID": "T322",
    "User_ID": "4582",
    "Transaction_Amount": 1881.9,
    "Transaction_Type": "Bill Payment",
    "Time_of_Transaction": 17.0,
    "Device_Used": "Tablet",
    "Location": "Los Angeles",
    "Previous_Fraudulent_Transactions":1,
    "Account_Age": 19,
    "Number_of_Transactions_Last_24H": 9,
    "Payment_Method": "Debit Card",
    "Fraudulent": "1"
}

Exemple 3 : 
{
    "Transaction_ID": "T324",
    "User_ID": "3191",
    "Transaction_Amount": 3040.03,
    "Transaction_Type": "POS Payment",
    "Time_of_Transaction": 4.0,
    "Device_Used": "Desktop",
    "Location": "Boston",
    "Previous_Fraudulent_Transactions":3,
    "Account_Age": 11,
    "Number_of_Transactions_Last_24H": 2,
    "Payment_Method": "Debit Card",
    "Fraudulent": "1"
}

Exemple 4 : 
{
    "Transaction_ID": "T482",
    "User_ID": "1180",
    "Transaction_Amount": 3901.23,
    "Transaction_Type": "Bank Transfert",
    "Time_of_Transaction": 23.0,
    "Device_Used": "Mobile",
    "Location": "Houston",
    "Previous_Fraudulent_Transactions":2,
    "Account_Age": 91,
    "Number_of_Transactions_Last_24H": 14,
    "Payment_Method": "Debit Card",
    "Fraudulent": "1"
}

Exemple 5 : 
{
    "Transaction_ID": "T497",
    "User_ID": "3065",
    "Transaction_Amount": 31.77,
    "Transaction_Type": "Bill Payment",
    "Time_of_Transaction": 23.0,
    "Device_Used": "Mobile",
    "Location": "Houston",
    "Previous_Fraudulent_Transactions":2,
    "Account_Age": 34,
    "Number_of_Transactions_Last_24H": 10,
    "Payment_Method": "Net Banking",
    "Fraudulent": "1"
}

Exemple 6 : 
{
    "Transaction_ID": "T498",
    "User_ID": "2816",
    "Transaction_Amount": 1406.52,
    "Transaction_Type": "Bill Payment",
    "Time_of_Transaction": 17.0,
    "Device_Used": "Tablet",
    "Location": "New York",
    "Previous_Fraudulent_Transactions":2,
    "Account_Age": 19,
    "Number_of_Transactions_Last_24H": 3,
    "Payment_Method": "Debit Card",
    "Fraudulent": "1"
}

Exemple 7 : 
{
    "Transaction_ID": "T615",
    "User_ID": "2012",
    "Transaction_Amount": 1488.1,
    "Transaction_Type": "ATM Withdrawal",
    "Time_of_Transaction": 22.0,
    "Device_Used": "Tablet",
    "Location": "Houston",
    "Previous_Fraudulent_Transactions":2,
    "Account_Age": 37,
    "Number_of_Transactions_Last_24H": 4,
    "Payment_Method": "Credit Card",
    "Fraudulent": "1"
}

### Génère maintenant un NOUVEAU cas de fraude différent des exemples :
"""
}
]

response = client.chat_completion(
    messages=messages,
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    max_tokens=300,
    temperature=0.5
)

generated_text = response.choices[0].message.content

# Extraire le JSON
try:
    json_start = generated_text.find('{')
    json_end = generated_text.rfind('}') + 1
    if json_start != -1 and json_end > json_start:
        json_str = generated_text[json_start:json_end]
        fraud_case = json.loads(json_str)
        print("\n=== Texte généré (JSON parsé) ===")
        print(json.dumps(fraud_case, indent=2, ensure_ascii=False))
    else:
        print("\n⚠️  Pas de JSON trouvé dans la réponse")
except Exception as e:
    print(f"\n❌ Erreur: {e}")

On va maintenant faire ces calls API afin de modifier notre dataset Fraud Detection

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Fraud Detection Dataset.csv")

# Détection des lignes incomplètes
rows_with_nan = df[df.isna().any(axis=1)]
print(f"{len(rows_with_nan)} lignes à compléter")

def row_to_json(row):
# cette fonction sert à remplacer Nan par null dans le dataset pour que le LLM comprenne
    return {
        k: (None if pd.isna(v) else v)
        for k, v in row.items()
    }

In [ ]:
PROMPT_TEMPLATE = """
Tu es un expert en détection de fraude à l’assurance.

Ta tâche est de COMPLÉTER les champs manquants d’un enregistrement existant.
- Ne modifie PAS les champs déjà renseignés
- Valeurs réalistes et cohérentes
- Raisonne étape par étape mais ne montre PAS ton raisonnement
- Réponds uniquement avec un JSON valide, sans texte autour

### Exemple

Entrée :
{{
    "Transaction_ID": "T3",
    "User_ID": "1860",
    "Transaction_Amount": 2395.02,
    "Transaction_Type": "ATM Withdrawal",
    "Time_of_Transaction": null,
    "Device_Used": "Mobile",
    "Location": "null",
    "Previous_Fraudulent_Transactions":3,
    "Account_Age": 115,
    "Number_of_Transactions_Last_24H": 9,
    "Payment_Method": "null",
    "Fraudulent": "0"
}}

Sortie :
{{
    "Transaction_ID": "T3",
    "User_ID": "1860",
    "Transaction_Amount": 2395.02,
    "Transaction_Type": "ATM Withdrawal",
    "Time_of_Transaction": 11.47734,
    "Device_Used": "Mobile",
    "Location": "Boston",
    "Previous_Fraudulent_Transactions":3,
    "Account_Age": 115,
    "Number_of_Transactions_Last_24H": 9,
    "Payment_Method": "Credit Card",
    "Fraudulent": "0"
}}

Entrée :
{{
    "Transaction_ID": "T112",
    "User_ID": "1455",
    "Transaction_Amount": null,
    "Transaction_Type": "Online Purchase",
    "Time_of_Transaction": 18.0,
    "Device_Used": "Mobile",
    "Location": "null",
    "Previous_Fraudulent_Transactions":1,
    "Account_Age": 116,
    "Number_of_Transactions_Last_24H": 11,
    "Payment_Method": "UPI",
    "Fraudulent": "0"
}}

Sortie :
{{
    "Transaction_ID": "T112",
    "User_ID": "1455",
    "Transaction_Amount": 2996.242497,
    "Transaction_Type": "Online Purchase",
    "Time_of_Transaction": 18.0,
    "Device_Used": "Mobile",
    "Location": "Seattle",
    "Previous_Fraudulent_Transactions": 1,
    "Account_Age": 116,
    "Number_of_Transactions_Last_24H": 11,
    "Payment_Method": "UPI",
    "Fraudulent": "0"
}}

Voici l'enregistrement à compléter :

{ROW_JSON}
"""



In [ ]:

from huggingface_hub import InferenceClient
import json
import os

os.environ["HF_TOKEN"] = "hf_xxxxxxxxx"
client = InferenceClient(token=os.getenv("HF_TOKEN"))


def complete_row_with_llm(row_dict):
    messages = [{
        "role": "user",
        "content": PROMPT_TEMPLATE.format(
            ROW_JSON=json.dumps(row_dict, ensure_ascii=False)
        )
    }]

    response = client.chat_completion(
        messages=messages,
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        temperature=0.3,
        max_tokens=300
    )

    text = response.choices[0].message.content

    json_start = text.find('{')
    json_end = text.rfind('}') + 1
    return json.loads(text[json_start:json_end])

In [ ]:
# on boucle sur tout le dataset

completed_rows = []

for idx, row in df.iterrows():
    if idx >= 50: #on n'a pas assez de jetons pour le faire sur toutes les lignes du dataset
        break
    if row.isna().any():
        print(f"🔧 Complétion ligne {idx} – Transaction_ID={row['Transaction_ID']}")
        row_json = row_to_json(row)
        completed = complete_row_with_llm(row_json)
        completed_rows.append(completed)
    else:
        completed_rows.append(row.to_dict())

df_completed = pd.DataFrame(completed_rows)

# vérification que toutes les Na ont bien été complétées
print(df_completed.isna().sum())

# Il faut maintenant voir comment on fait les métriques car on n'a qu'un bout du dataset complété à cause des tokens...
# Il faut aussi voir comment générer suffisamment de données frauduleuses sans épuiser les tokens...